In [9]:
#!/usr/bin/env python3

import os
import csv
import json
import re
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt


def safe_name(path):
    """
    Convert a relative path into a safe filename.
    """
    return re.sub(r"[^A-Za-z0-9_.-]+", "__", str(path))


def flatten_config(config):
    """
    Flatten a nested JSON dictionary.

    Example:
        config["params"]["alpha"] -> flat["alpha"]
        config["params"]["alpha"] -> flat["params.alpha"]

    Both versions are stored. The prefixed version is safer.
    """
    flat = {}

    for group, values in config.items():
        if isinstance(values, dict):
            for key, value in values.items():
                flat[f"{group}.{key}"] = value

                # Also keep the short name if it does not already exist.
                if key not in flat:
                    flat[key] = value
        else:
            flat[group] = values

    return flat


def read_json_near_h5(h5_path):
    """
    Try to read a JSON file in the same folder as data.h5.
    """
    sim_dir = h5_path.parent
    json_files = sorted(sim_dir.glob("*.json"))

    if not json_files:
        return {}

    with open(json_files[0], "r") as f:
        config = json.load(f)

    return flatten_config(config)


def get_last_frame(h5_path, field):
    """
    Read last frame from an HDF5 file.

    Default options:
        field = "psi"   -> solution/psi_hist
        field = "p_mag" -> sqrt(px_hist^2 + py_hist^2)
        field = "q_mag" -> sqrt(qx_hist^2 + qy_hist^2)

    You can also pass a direct dataset name, e.g.
        field = "psi_hist"
        field = "qx_hist"
    """

    with h5py.File(h5_path, "r") as f:
        if "solution" in f:
            sol = f["solution"]
        else:
            sol = f

        if field == "psi":
            frame = sol["psi_hist"][-1]

        elif field in ["p_mag", "pmag"]:
            px = sol["px_hist"][-1]
            py = sol["py_hist"][-1]
            frame = np.sqrt(px**2 + py**2)

        elif field in ["q_mag", "qmag"]:
            qx = sol["qx_hist"][-1]
            qy = sol["qy_hist"][-1]
            frame = np.sqrt(qx**2 + qy**2)

        else:
            # Direct dataset name
            if field in sol:
                ds = sol[field]
            elif field in f:
                ds = f[field]
            else:
                raise KeyError(f"Dataset '{field}' not found in {h5_path}")

            if ds.ndim >= 3:
                frame = ds[-1]
            else:
                frame = ds[...]

        extent = None
        if "grid" in f and "x" in f["grid"] and "y" in f["grid"]:
            x = f["grid"]["x"][...]
            y = f["grid"]["y"][...]

            extent = [x.min(), x.max(), y.min(), y.max()]

    return np.array(frame), extent


def make_title(h5_path, params):
    """
    Make a compact title using useful parameters if available.
    """
    useful = ["lam_p", "noise_mag", "v0", "t_dec"]

    parts = []
    for key in useful:
        if key in params:
            parts.append(f"{key}={params[key]}")

    rel_name = h5_path.parent.name

    if parts:
        return rel_name + "\n" + ", ".join(parts)

    return rel_name


def save_frame_png(frame, extent, out_path, title, cmap):
    """
    Save one frame as a PNG image.
    """
    frame = np.ma.masked_invalid(frame)

    fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)

    imshow_kwargs = {
        "origin": "lower",
        "aspect": "equal",
        "cmap": cmap,
    }

    if extent is not None:
        imshow_kwargs["extent"] = extent

    im = ax.imshow(frame, **imshow_kwargs)

    ax.set_title(title, fontsize=9)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    fig.colorbar(im, ax=ax)

    fig.savefig(out_path, dpi=200)
    plt.close(fig)


root_dir = "outputs/dyn_no_q_sweep"
out_dir = os.path.join(root_dir, "last_frames")
os.makedirs(out_dir, exist_ok=True)

cmap = "viridis"

h5_files = sorted(Path(root_dir).rglob("data.h5"))


rows = []

for h5_path in h5_files:
    try:
        rel = h5_path.parent.relative_to(root_dir)
        out_name = safe_name(rel) + f"__psi_last.png"
        out_path = os.path.join(out_dir, out_name)

        params = read_json_near_h5(h5_path)

        frame, extent = get_last_frame(h5_path, "psi")
        title = make_title(h5_path, params)

        save_frame_png(
            frame=frame,
            extent=extent,
            out_path=out_path,
            title=title,
            cmap=cmap,
        )

        nan_fraction = np.isnan(frame).mean()

        row = {
            "sim_dir": str(rel),
            "h5_path": str(h5_path),
            "png_path": str(out_path),
            "field": "psi",
            "nan_fraction": nan_fraction,
        }

        # Add JSON parameters to the index table
        row.update(params)

        rows.append(row)

        print(f"[OK] {rel} -> {out_path.name}")

    except Exception as e:
        print(f"[ERROR] {h5_path}: {e}")


    all_keys = sorted({key for row in rows for key in row.keys()})


    print()
    print(f"Saved {len(rows)} frames in:")
    print(out_dir)
    print(f"Index saved in:")


[ERROR] outputs/dyn_no_q_sweep/sim_data_20260702_112009_it_1060456429/data.h5: 'str' object has no attribute 'name'

Saved 1 frames in:
outputs/dyn_no_q_sweep/last_frames
Index saved in:
[ERROR] outputs/dyn_no_q_sweep/sim_data_20260702_112009_it_1062548554/data.h5: 'str' object has no attribute 'name'

Saved 2 frames in:
outputs/dyn_no_q_sweep/last_frames
Index saved in:
[ERROR] outputs/dyn_no_q_sweep/sim_data_20260702_112009_it_1079743915/data.h5: 'str' object has no attribute 'name'

Saved 3 frames in:
outputs/dyn_no_q_sweep/last_frames
Index saved in:
[ERROR] outputs/dyn_no_q_sweep/sim_data_20260702_112009_it_1262628704/data.h5: 'str' object has no attribute 'name'

Saved 4 frames in:
outputs/dyn_no_q_sweep/last_frames
Index saved in:
[ERROR] outputs/dyn_no_q_sweep/sim_data_20260702_112009_it_1818693446/data.h5: 'str' object has no attribute 'name'

Saved 5 frames in:
outputs/dyn_no_q_sweep/last_frames
Index saved in:
[ERROR] outputs/dyn_no_q_sweep/sim_data_20260702_112009_it_193861